## For [train, train_preserved_imputed, train_clean_imputed, train_clean] 
## 1. Evaluate on the given dataset.
## 2. Transform skewed variables -> evluate.
## 3. Categorize weather features -> evaluate.
## 4. Add Weekday variable -> evaluate.
## 5. Transform time variables to cyclic -> evaluate.
## Choose the best.

In [1]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_train_preserved_imputed = pd.read_csv("train_preserved_imputed.csv")
df_train_clean_imputed = pd.read_csv("train_clean_imputed.csv")
df_train_clean = pd.read_csv("train_clean.csv")

In [2]:
import numpy as np

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import StratifiedKFold, cross_val_score


# ============================================================
# COMMON CONFIGURATION
# ============================================================

TARGET = "Demand_Category"

DATASETS = {
    "train": df_train.copy(),
    "train_preserved_imputed": df_train_preserved_imputed.copy(),
    "train_clean_imputed": df_train_clean_imputed.copy(),
    "train_clean": df_train_clean.copy(),
}

BASE_NUMERIC_FEATURES = [
    "Hour",
    "Temperature",
    "Humidity",
    "Wind speed",
    "Visibility",
    "Dew point temperature",
    "Solar Radiation",
    "Rainfall",
    "Snowfall",
]

BASE_CATEGORICAL_FEATURES = [
    "Seasons",
    "Holiday",
    "Functioning Day",
]

WEATHER_NUMERIC_FEATURES = [
    "Wind speed",
    "Solar Radiation",
    "Rainfall",
    "Snowfall",
    "Visibility",
]

CV = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [3]:
def build_preprocessor(numeric_features, categorical_features, X_columns):
    """
    Build preprocessing pipeline.

    Numerical:
        Iterative imputation -> standardization

    Categorical:
        Most-frequent imputation -> one-hot encoding
    """

    num_cols = [
        col for col in numeric_features
        if col in X_columns
    ]

    cat_cols = [
        col for col in categorical_features
        if col in X_columns
    ]

    numerical_transformer = Pipeline(
        steps=[
            (
                "imputer",
                IterativeImputer(
                    max_iter=10,
                    random_state=42
                )
            ),
            (
                "scaler",
                StandardScaler()
            )
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="most_frequent")
            ),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore",
                    drop="first"
                )
            )
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                numerical_transformer,
                num_cols
            ),
            (
                "cat",
                categorical_transformer,
                cat_cols
            )
        ],
        remainder="drop"
    )

    return preprocessor


def evaluate_logistic_regression(
    df,
    numeric_features,
    categorical_features,
    experiment_name=""
):
    """
    Evaluate Logistic Regression using identical
    Stratified 5-fold CV and macro-F1.
    """

    X = df.drop(
        columns=[
            TARGET,
            "Kaggle_ID",
            "Record_id"
        ],
        errors="ignore"
    )

    y = df[TARGET]

    preprocessor = build_preprocessor(
        numeric_features=numeric_features,
        categorical_features=categorical_features,
        X_columns=X.columns
    )

    model = LogisticRegression(
        max_iter=2000,
        random_state=42
    )

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", model)
        ]
    )

    scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=CV,
        scoring="f1_macro",
        n_jobs=-1
    )

    result = {
        "Experiment": experiment_name,
        "Mean Macro F1": scores.mean(),
        "Std Macro F1": scores.std(),
        "Fold 1": scores[0],
        "Fold 2": scores[1],
        "Fold 3": scores[2],
        "Fold 4": scores[3],
        "Fold 5": scores[4],
    }

    print(
        f"{experiment_name:<45} "
        f"Macro-F1 = {scores.mean():.4f} "
        f"(± {scores.std():.4f})"
    )

    return result

In [4]:
def add_skew_transformations(df):
    """
    Add transformed versions of skewed weather variables.

    Original variables are retained.
    """

    out = df.copy()

    # Right-skewed non-negative variables
    right_skewed = [
        "Wind speed",
        "Solar Radiation",
        "Rainfall",
        "Snowfall"
    ]

    for col in right_skewed:
        if col in out.columns:
            # Negative values are physically invalid for these variables.
            # Turn them into NaN so the pipeline can impute them.
            values = out[col].mask(out[col] < 0)

            out[f"{col}_log1p"] = np.log1p(values)

    # Visibility: your original power transformation
    if "Visibility" in out.columns:
        values = out["Visibility"].mask(
            out["Visibility"] < 0
        )

        out["Visibility_pow3"] = values ** 3

    return out

In [5]:
def add_weather_categories(df):
    """
    Add categorical versions of heavily skewed weather variables.
    Original continuous features are retained.
    """

    out = df.copy()

    # Visibility
    if "Visibility" in out.columns:
        bins = [-np.inf, 500, 1500, np.inf]

        out["Visibility_Cat"] = pd.cut(
            out["Visibility"],
            bins=bins,
            labels=["low", "medium", "high"]
        )

    # Solar radiation
    if "Solar Radiation" in out.columns:
        bins = [-np.inf, 0.1, 1.5, np.inf]

        out["Solar_Radiation_Cat"] = pd.cut(
            out["Solar Radiation"],
            bins=bins,
            labels=["none", "low", "high"]
        )

    # Rainfall
    if "Rainfall" in out.columns:
        bins = [-np.inf, 0.1, 2.0, np.inf]

        out["Rainfall_Cat"] = pd.cut(
            out["Rainfall"],
            bins=bins,
            labels=["none", "light", "heavy"]
        )

    # Snowfall
    if "Snowfall" in out.columns:
        bins = [-np.inf, 0.1, 1.0, np.inf]

        out["Snowfall_Cat"] = pd.cut(
            out["Snowfall"],
            bins=bins,
            labels=["none", "light", "heavy"]
        )

    return out

In [6]:
def add_weekday(df):
    """
    Add weekday derived from Date.

    Monday = 0
    Sunday = 6
    """

    out = df.copy()

    if "Date" in out.columns:

        date_parsed = pd.to_datetime(
            out["Date"],
            dayfirst=True,
            errors="coerce"
        )

        out["Weekday"] = date_parsed.dt.dayofweek

    return out

In [7]:
def add_month(df):
    out = df.copy()

    if "Date" in out.columns:
        date_parsed = pd.to_datetime(
            out["Date"],
            dayfirst=True,
            errors="coerce"
        )

        out["Month"] = date_parsed.dt.month

    return out

In [8]:
def add_all_cyclic_time_features(df):
    """
    Add cyclic representations for:
        Hour
        Weekday
        Month
        Seasons
    """

    out = df.copy()

    # --------------------------------------------------------
    # Hour
    # --------------------------------------------------------
    if "Hour" in out.columns:
        out["Hour_sin"] = np.sin(
            2 * np.pi * out["Hour"] / 24
        )
        out["Hour_cos"] = np.cos(
            2 * np.pi * out["Hour"] / 24
        )

    # --------------------------------------------------------
    # Weekday
    # --------------------------------------------------------
    if "Weekday" in out.columns:
        out["Weekday_sin"] = np.sin(
            2 * np.pi * out["Weekday"] / 7
        )
        out["Weekday_cos"] = np.cos(
            2 * np.pi * out["Weekday"] / 7
        )

    # --------------------------------------------------------
    # Month
    # --------------------------------------------------------
    if "Month" in out.columns:
        out["Month_sin"] = np.sin(
            2 * np.pi * (out["Month"] - 1) / 12
        )
        out["Month_cos"] = np.cos(
            2 * np.pi * (out["Month"] - 1) / 12
        )

    # --------------------------------------------------------
    # Season
    # --------------------------------------------------------
    if "Seasons" in out.columns:

        season_mapping = {
            "Winter": 0,
            "Spring": 1,
            "Summer": 2,
            "Autumn": 3
        }

        season_num = out["Seasons"].map(season_mapping)

        out["Season_sin"] = np.sin(
            2 * np.pi * season_num / 4
        )

        out["Season_cos"] = np.cos(
            2 * np.pi * season_num / 4
        )

    return out

In [9]:
SKEW_NUMERIC_FEATURES = BASE_NUMERIC_FEATURES + [
    "Wind speed_log1p",
    "Solar Radiation_log1p",
    "Rainfall_log1p",
    "Snowfall_log1p",
    "Visibility_pow3",
]

WEATHER_CATEGORY_FEATURES = BASE_CATEGORICAL_FEATURES + [
    "Visibility_Cat",
    "Solar_Radiation_Cat",
    "Rainfall_Cat",
    "Snowfall_Cat",
]

WEEKDAY_NUMERIC_FEATURES = BASE_NUMERIC_FEATURES + [
    "Weekday"
]

CYCLIC_NUMERIC_FEATURES = [
    "Temperature",
    "Humidity",
    "Wind speed",
    "Visibility",
    "Dew point temperature",
    "Solar Radiation",
    "Rainfall",
    "Snowfall",
    "Hour_sin",
    "Hour_cos",
]

In [10]:
results = []

for dataset_name, original_df in DATASETS.items():

    print("\n")
    print("=" * 80)
    print(f"DATASET: {dataset_name}")
    print("=" * 80)

    # --------------------------------------------------------
    # 1. GIVEN DATASET / BASELINE
    # --------------------------------------------------------

    df_exp = original_df.copy()

    results.append(
        evaluate_logistic_regression(
            df=df_exp,
            numeric_features=BASE_NUMERIC_FEATURES,
            categorical_features=BASE_CATEGORICAL_FEATURES,
            experiment_name=f"{dataset_name} | 1. Baseline"
        )
    )

    # --------------------------------------------------------
    # 2. TRANSFORM SKEWED VARIABLES
    # --------------------------------------------------------

    df_exp = add_skew_transformations(df_exp)

    results.append(
        evaluate_logistic_regression(
            df=df_exp,
            numeric_features=SKEW_NUMERIC_FEATURES,
            categorical_features=BASE_CATEGORICAL_FEATURES,
            experiment_name=f"{dataset_name} | 2. Skew transformations"
        )
    )

    # --------------------------------------------------------
    # 3. CATEGORIZE WEATHER FEATURES
    # --------------------------------------------------------

    df_exp = add_weather_categories(df_exp)

    results.append(
        evaluate_logistic_regression(
            df=df_exp,
            numeric_features=BASE_NUMERIC_FEATURES + ["Wind speed_log1p"],
            categorical_features=WEATHER_CATEGORY_FEATURES,
            experiment_name=f"{dataset_name} | 3. + Weather categories"
        )
    )
            
    # --------------------------------------------------------
    # 4. ADD WEEKDAY
    # --------------------------------------------------------

    df_exp = add_weekday(df_exp)

    categorical_with_weekday = BASE_CATEGORICAL_FEATURES + [
        "Weekday"
    ]

    results.append(
        evaluate_logistic_regression(
            df=df_exp,
            numeric_features=SKEW_NUMERIC_FEATURES,
            categorical_features=categorical_with_weekday,
            experiment_name=f"{dataset_name} | 4. + Weekday"
        )
    )


    # --------------------------------------------------------
    # 5. TRANSFORM TIME VARIABLES TO CYCLIC
    # --------------------------------------------------------

    df_exp = add_weekday(df_exp)
    df_exp = add_month(df_exp)
    df_exp = add_all_cyclic_time_features(df_exp)

    numeric_cyclic = SKEW_NUMERIC_FEATURES + [
        "Weekday_sin",
        "Weekday_cos",
        "Month_sin",
        "Month_cos",
        "Hour_sin",
        "Hour_cos",
        "Season_sin",
        "Season_cos",
    ]

    categorical_cyclic = [
        "Holiday",
        "Functioning Day"
    ]

    results.append(
        evaluate_logistic_regression(
            df=df_exp,
            numeric_features=numeric_cyclic,
            categorical_features=categorical_cyclic,
            experiment_name=(
                f"{dataset_name} | "
                "5. Cyclic time: Hour+Weekday+Month+Season"
            )
        )
    )



DATASET: train
train | 1. Baseline                           Macro-F1 = 0.6539 (± 0.0150)
train | 2. Skew transformations               Macro-F1 = 0.6599 (± 0.0160)
train | 3. + Weather categories               Macro-F1 = 0.6544 (± 0.0095)
train | 4. + Weekday                          Macro-F1 = 0.6598 (± 0.0151)
train | 5. Cyclic time: Hour+Weekday+Month+Season Macro-F1 = 0.6888 (± 0.0096)


DATASET: train_preserved_imputed
train_preserved_imputed | 1. Baseline         Macro-F1 = 0.7219 (± 0.0112)
train_preserved_imputed | 2. Skew transformations Macro-F1 = 0.7231 (± 0.0095)
train_preserved_imputed | 3. + Weather categories Macro-F1 = 0.7239 (± 0.0100)
train_preserved_imputed | 4. + Weekday        Macro-F1 = 0.7266 (± 0.0092)
train_preserved_imputed | 5. Cyclic time: Hour+Weekday+Month+Season Macro-F1 = 0.7376 (± 0.0084)


DATASET: train_clean_imputed
train_clean_imputed | 1. Baseline             Macro-F1 = 0.7243 (± 0.0100)
train_clean_imputed | 2. Skew transformations Macro-F1 = 0

In [11]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "Mean Macro F1",
    ascending=False
).reset_index(drop=True)

display(
    results_df[
        [
            "Experiment",
            "Mean Macro F1",
            "Std Macro F1"
        ]
    ]
)

,Experiment,Mean Macro F1,Std Macro F1
0,train_clean | 5. Cyclic time: Hour+Weekday+Mon...,0.739061,0.014997
1,train_clean_imputed | 5. Cyclic time: Hour+Wee...,0.737990,0.009468
2,train_preserved_imputed | 5. Cyclic time: Hour...,0.737620,0.008388
3,train_clean | 3. + Weather categories,0.736213,0.016035
4,train_clean | 4. + Weekday,0.735585,0.014887
5,train_clean | 2. Skew transformations,0.732448,0.016823
6,train_clean_imputed | 4. + Weekday,0.728066,0.009991
7,train_clean_imputed | 2. Skew transformations,0.727888,0.010212
8,train_preserved_imputed | 4. + Weekday,0.726583,0.009239
9,train_clean_imputed | 3. + Weather categories,0.724458,0.010380


In [12]:
best_per_dataset = (
    results_df
    .assign(
        Dataset=results_df["Experiment"].str.split(" | ").str[0]
    )
    .loc[
        lambda df: df.groupby("Dataset")["Mean Macro F1"]
        .transform("max") == df["Mean Macro F1"]
    ]
)

display(
    best_per_dataset[
        [
            "Dataset",
            "Experiment",
            "Mean Macro F1",
            "Std Macro F1"
        ]
    ]
)

,Dataset,Experiment,Mean Macro F1,Std Macro F1
0,train_clean,train_clean | 5. Cyclic time: Hour+Weekday+Mon...,0.739061,0.014997
1,train_clean_imputed,train_clean_imputed | 5. Cyclic time: Hour+Wee...,0.737990,0.009468
2,train_preserved_imputed,train_preserved_imputed | 5. Cyclic time: Hour...,0.737620,0.008388
15,train,train | 5. Cyclic time: Hour+Weekday+Month+Season,0.688839,0.009577


In [13]:
best_result = results_df.iloc[0]

print("BEST EXPERIMENT")
print("=" * 50)
print("Experiment:", best_result["Experiment"])
print("Mean Macro F1:", round(best_result["Mean Macro F1"], 4))
print("Std Macro F1:", round(best_result["Std Macro F1"], 4))

BEST EXPERIMENT
Experiment: train_clean | 5. Cyclic time: Hour+Weekday+Month+Season
Mean Macro F1: 0.7391
Std Macro F1: 0.015


In [14]:
results_table = results_df.copy()

results_table["Dataset"] = (
    results_table["Experiment"]
    .str.split(" | ")
    .str[0]
)

results_table["Step"] = (
    results_table["Experiment"]
    .str.split(" | ")
    .str[1]
)

pivot = results_table.pivot(
    index="Dataset",
    columns="Step",
    values="Mean Macro F1"
)

display(
    pivot.round(4)
)

ValueError: Index contains duplicate entries, cannot reshape